# Online Retail II — Cleaning (Plan D) — v3

**v3 fixes (external + independent review):** ① return netting is now **time-constrained** (a return only offsets purchases made *before* it — no future-netting); ② `clean_lines.pkl` now = the **netted, Customer-ID-only** line table (consistent with events; no-CID lines ~13% of revenue are excluded here but preserved in the raw pkl); ③ removed MIN_PURCHASES / pooled-gap decisions from here (they live in EDA).

**Outputs:** `processed/clean_lines.pkl` (netted CID lines), `processed/returns.pkl`, `processed/events.pkl`.


## Setup


In [1]:
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np, pandas as pd
from pathlib import Path
from collections import defaultdict
pd.set_option("display.max_columns", 50)


## Constants


In [2]:
DATA_DIR  = Path("../Dataset/UCI")
XLSX_PATH = DATA_DIR / "online_retail_II.xlsx"
RAW_PKL   = DATA_DIR / "online_retail_II_raw.pkl"
PROC_DIR  = DATA_DIR / "processed"; PROC_DIR.mkdir(exist_ok=True)
KEY="Customer ID"; INVOICE="Invoice"; DATE="InvoiceDate"; QTY="Quantity"; PRICE="Price"; STOCK="StockCode"
STR_COLS=["Invoice","StockCode","Description","Country"]


## 1. Load (pickle cache, else read both xlsx sheets)


In [3]:
def load_raw():
    if RAW_PKL.exists():
        return pd.read_pickle(RAW_PKL)
    sheets = pd.read_excel(XLSX_PATH, sheet_name=None, engine="openpyxl")
    frames=[]
    for name,d in sheets.items():
        d=d.copy(); d["source_sheet"]=name; frames.append(d)
    raw=pd.concat(frames, ignore_index=True)
    for c in STR_COLS: raw[c]=raw[c].astype("string")
    raw.to_pickle(RAW_PKL); return raw

raw = load_raw()
raw[DATE] = raw[DATE].astype("datetime64[ns]")
raw[KEY]  = raw[KEY].astype("Int64")
print("shape:", raw.shape); print("cols :", raw.columns.tolist())
raw.head()


shape: (1067371, 9)
cols : ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,Year 2009-2010


## 2. First look — null rates (table)


In [4]:
display(pd.DataFrame({"n_missing": raw.isna().sum(),
                      "missing_%": raw.isna().mean().mul(100).round(2),
                      "dtype": raw.dtypes.astype(str)}))
inv=raw[INVOICE].astype(str)
print("cancellation (C) rows %:", round(inv.str.startswith("C").mean()*100,2))
print("rows w/o Customer ID %:", round(raw[KEY].isna().mean()*100,2))
print("date range:", raw[DATE].min(), "->", raw[DATE].max())


,n_missing,missing_%,dtype
Invoice,0,0.00,object
StockCode,0,0.00,object
Description,4382,0.41,object
Quantity,0,0.00,int64
InvoiceDate,0,0.00,datetime64[ns]
Price,0,0.00,float64
Customer ID,243007,22.77,Int64
Country,0,0.00,object
source_sheet,0,0.00,object


cancellation (C) rows %: 1.83
rows w/o Customer ID %: 22.77
date range: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


## 3. Drop duplicate rows (business-level, excl. source_sheet)


In [5]:
BIZ=[c for c in raw.columns if c!="source_sheet"]
before=len(raw); df=raw.drop_duplicates(subset=BIZ).copy()
print(f"dropped duplicate rows: {before-len(df)} -> {len(df)}")


dropped duplicate rows: 34335 -> 1033036


## 4. Split cancellations / returns ('C' invoices)


In [6]:
is_cancel=df[INVOICE].astype(str).str.startswith("C")
returns=df[is_cancel].copy(); buy=df[~is_cancel].copy()
print("cancellation/return rows:", len(returns), "| purchase rows:", len(buy))


cancellation/return rows: 19104 | purchase rows: 1013932


## 5. Filter non-product StockCodes (^\d{5})


In [7]:
is_prod=buy[STOCK].astype(str).str.match(r"^\d{5}")
print("non-product rows removed:", int((~is_prod).sum()))
buy=buy[is_prod].copy()


non-product rows removed: 4790


## 6. Valid purchases + line revenue


In [8]:
buy=buy[(buy[QTY]>0)&(buy[PRICE]>0)].copy()
buy["line_revenue"]=buy[QTY]*buy[PRICE]
print("valid purchase rows (incl no-CID):", len(buy))


valid purchase rows (incl no-CID): 1003214


## 7. Keep Customer-ID rows for the sequence table
No-CID lines (~13% of revenue) are dropped here (can't attribute to a customer); the full record stays in the raw pkl.


In [9]:
seq=buy.dropna(subset=[KEY]).copy(); seq[KEY]=seq[KEY].astype("int64")
print("sequence rows (pre-net):", len(seq), "| customers:", seq[KEY].nunique())


sequence rows (pre-net): 776577 | customers: 5852


## 7b. Net returns against purchases — TIME-CONSTRAINED
**Fix:** a return only offsets purchases dated **before** the return (LIFO among those priors). Returns with no prior purchase are ignored — we never net a purchase that happened after the return.


In [10]:
r=returns.dropna(subset=[KEY]).copy(); r[KEY]=r[KEY].astype("int64"); r["ret_qty"]=r[QTY].abs()
seq=seq.sort_values([KEY,STOCK,DATE]).reset_index(drop=True)
seq["net_qty"]=seq[QTY].astype(float)

purch=defaultdict(list)
for pos,(k,s,d) in enumerate(zip(seq[KEY],seq[STOCK],seq[DATE])): purch[(k,s)].append((d,pos))
rets=defaultdict(list)
for k,s,d,q in zip(r[KEY],r[STOCK],r[DATE],r["ret_qty"]): rets[(k,s)].append((d,float(q)))

tot_ret=float(r["ret_qty"].sum()); unmatched=0.0; u_noprior=0.0; u_over=0.0
for key,rl in rets.items():
    priors=purch.get(key)
    for rdate,rq in sorted(rl):                           # returns in date order
        elig=[p for p in priors if p[0]<rdate] if priors else []
        if not elig:
            unmatched+=rq; u_noprior+=rq; continue        # no earlier purchase to offset (pre-window / anonymous)
        for d,pos in sorted(elig, reverse=True):          # prior purchases, newest first
            if rq<=0: break
            take=min(seq.at[pos,"net_qty"],rq); seq.at[pos,"net_qty"]-=take; rq-=take
        if rq>0: unmatched+=rq; u_over+=rq                # returned more than any prior purchase

seq_net=seq[seq["net_qty"]>0].copy()
seq_net[QTY]=seq_net["net_qty"].astype(int)
seq_net["line_revenue"]=seq_net[QTY]*seq_net[PRICE]
print("lines after time-constrained netting:", len(seq_net), "| dropped fully-returned lines:", len(seq)-len(seq_net))
print(f"return qty: total {int(tot_ret)} | matched {int(tot_ret-unmatched)} ({(1-unmatched/tot_ret)*100:.1f}%) | UNMATCHED {int(unmatched)} ({unmatched/tot_ret*100:.1f}%: no-prior {int(u_noprior)}, over-return {int(u_over)})")


lines after time-constrained netting: 770551 | dropped fully-returned lines: 6026
return qty: total 472976 | matched 450047 (95.2%) | UNMATCHED 22929 (4.8%: no-prior 21168, over-return 1761)


## 8. Aggregate to invoice-level events (sequence unit)


In [11]:
events=(seq_net.groupby([KEY,INVOICE],as_index=False)
            .agg(event_time=(DATE,"min"), n_items=(STOCK,"nunique"),
                 basket_qty=(QTY,"sum"), basket_value=("line_revenue","sum"))
            .sort_values([KEY,"event_time"]))
print("events:", len(events), "| customers:", events[KEY].nunique())
events.head()


events: 36316 | customers: 5839


,Customer ID,Invoice,event_time,n_items,basket_qty,basket_value
0,12346,499763,2010-03-02 13:08:00,5,5,27.05
1,12346,513774,2010-06-28 13:53:00,19,19,142.31
2,12347,529924,2010-10-31 14:20:00,40,509,611.53
3,12347,537626,2010-12-07 14:57:00,31,319,711.79
4,12347,542237,2011-01-26 14:30:00,29,315,475.39


## 8b. Integrity assertions


In [12]:
assert df.groupby(INVOICE)["source_sheet"].nunique().le(1).all(), "invoice spans two sheets"
assert (events["basket_qty"]>=events["n_items"]).all(), "basket_qty < n_items"
assert events[[KEY,INVOICE]].duplicated().sum()==0
assert events["basket_value"].ge(0).all()
print("integrity checks passed for", len(events), "events")


integrity checks passed for 36316 events


## 9. Save processed artifacts (pickle)


In [13]:
clean_lines = seq_net.drop(columns=[c for c in ["net_qty"] if c in seq_net.columns])
clean_lines.to_pickle(PROC_DIR/"clean_lines.pkl")   # netted, Customer-ID-only line table
returns.to_pickle(PROC_DIR/"returns.pkl")
events.to_pickle(PROC_DIR/"events.pkl")
print("clean_lines nulls:", int(clean_lines.isna().sum().sum()), "| events nulls:", int(events.isna().sum().sum()))
print("saved ->", [p.name for p in PROC_DIR.glob("*.pkl")])


clean_lines nulls: 0 | events nulls: 0
saved -> ['clean_lines.pkl', 'returns.pkl', 'gru_capacity_ablation.pkl', 'features_tabular.pkl', 'gru_test_scores.pkl', 'events.pkl', 'baseline_test_scores.pkl', 'baseline_results_table.pkl', 'gru_comparison_table.pkl']


## Decisions log (v3)

| step | result |
|---|---|
| raw rows | 1,067,371 |
| duplicate rows dropped (business-level) | 34,335 |
| cancellation/return rows | 19,104 |
| non-product rows removed | 4,790 |
| valid purchase rows (incl no-CID) | 1,003,214 |
| sequence rows w/ Customer ID (pre-net) | 776,577 |
| **time-constrained** netting: lines kept | 770,551 |
| fully-returned lines dropped | 6,026 |
| return qty matched / unmatched | 95.2% / 4.8% (unmatched = returns of pre-window or anonymous purchases; left-censoring) |
| invoice events | 36,316 |
| unique customers | 5,839 |

**Notes:** returns net only against *prior* purchases (no future-netting). `clean_lines.pkl` = netted Customer-ID-only lines (consistent with events); no-CID lines (~13% of revenue) excluded here, preserved in `raw` pkl. MIN_PURCHASES / inter-purchase-gap decisions live in the EDA notebook, not here. Dataset: UCI Online Retail II, CC BY 4.0.
